In [1]:
generation_model = "DDPM"

In [2]:
# Configuração para Jupyter
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np
from PIL import Image
import torch
import torchvision.transforms as transforms
from torchvision.models import resnet18, ResNet18_Weights
from sklearn.neighbors import BallTree
import warnings
warnings.filterwarnings('ignore')
import gc
import json

# ================= CONFIGURAÇÃO DOS PARÂMETROS =================
# Defina seus caminhos aqui
REAL_PATH = "DATASET54000"  # ← ALTERE AQUI
FAKE_PATH = str(generation_model + "-F64x4")  # ← ALTERE AQUI

# Parâmetros de execução
MAX_SAMPLES_PER_CLASS = 5400      # Número máximo de amostras por classe
BATCH_SIZE = 16     # Tamanho do batch
IMAGE_SIZE = 256         # Tamanho das imagens
OUTPUT_DIR = str(generation_model + "-RESULTS")  # Pasta para salvar resultados

# Verificar se CUDA está disponível
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dispositivo: {device}")

# ================= FUNÇÕES PARA ESTRUTURA POR SUBPASTAS =================

def get_subfolder_classes(root_path):
    """
    Obtém classes baseadas em subpastas - PARA ESTRUTURA OPÇÃO 1
    """
    classes = []
    class_images = {}

    # Listar apenas diretórios (subpastas)
    items = os.listdir(root_path)
    subdirs = [item for item in items if os.path.isdir(os.path.join(root_path, item))]

    print(f"Subpastas encontradas em {root_path}: {subdirs}")

    for class_name in subdirs:
        class_path = os.path.join(root_path, class_name)
        images = []

        # Listar arquivos de imagem na subpasta
        for img_name in os.listdir(class_path):
            if any(img_name.lower().endswith(ext) for ext in ['.jpg', '.jpeg', '.png', '.bmp', '.tiff']):
                images.append(os.path.join(class_path, img_name))

        if images:  # Só adicionar se tiver imagens
            classes.append(class_name)
            class_images[class_name] = images
            print(f"  Classe '{class_name}': {len(images)} imagens")

    return classes, class_images

class ImageFolderWithClasses:
    def __init__(self, root, transform=None, max_samples_per_class=None):
        self.root = root
        self.transform = transform
        self.max_samples_per_class = max_samples_per_class
        self.class_names, self.class_images = get_subfolder_classes(root)
        self.images, self.labels = self._prepare_data()
        self.class_to_idx = {cls: idx for idx, cls in enumerate(self.class_names)}
        print(f"Total de {len(self.images)} imagens em {len(self.class_names)} classes")

    def _prepare_data(self):
        images = []
        labels = []

        for class_idx, class_name in enumerate(self.class_names):
            class_image_list = self.class_images[class_name]

            # Limitar amostras por classe se especificado
            if self.max_samples_per_class and len(class_image_list) > self.max_samples_per_class:
                class_image_list = class_image_list[:self.max_samples_per_class]

            for img_path in class_image_list:
                images.append(img_path)
                labels.append(class_idx)

        return images, labels

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        try:
            image = Image.open(img_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
            return image, self.labels[idx]
        except Exception as e:
            print(f"Erro ao carregar {img_path}: {e}")
            # Retorna imagem preta em caso de erro
            return torch.zeros(3, IMAGE_SIZE, IMAGE_SIZE), self.labels[idx]

def extract_features_with_labels(dataloader, model, device, max_samples=None):
    features = []
    labels = []
    total_samples = 0

    with torch.no_grad():
        for batch, batch_labels in dataloader:
            batch = batch.to(device)
            feat = model(batch)
            if feat.dim() == 4:
                feat = feat.squeeze(-1).squeeze(-1)

            features.append(feat.cpu().numpy())
            labels.append(batch_labels.numpy())

            total_samples += batch.size(0)
            if max_samples and total_samples >= max_samples:
                break

            # Limpeza de memória periódica
            if len(features) % 10 == 0:
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

    features = np.concatenate(features, axis=0)
    labels = np.concatenate(labels, axis=0)

    if max_samples:
        features = features[:max_samples]
        labels = labels[:max_samples]

    return features, labels

def compute_prdc_for_class(real_features_class, fake_features_class, nearest_k=5):
    """
    Calcula PRDC para uma classe específica
    """
    n_real = len(real_features_class)
    n_fake = len(fake_features_class)

    if n_real < nearest_k or n_fake < nearest_k:
        print(f"  ⚠️  Amostras insuficientes: {n_real} real, {n_fake} fake (mínimo {nearest_k})")
        return 0.0, 0.0

    all_features = np.concatenate([real_features_class, fake_features_class], axis=0)

    # Usar BallTree para busca eficiente
    tree = BallTree(all_features, leaf_size=40)

    # Calcular precision
    precision_scores = []
    batch_size = min(100, n_real)

    for i in range(0, n_real, batch_size):
        end_idx = min(i + batch_size, n_real)
        real_batch = all_features[i:end_idx]

        distances, indices = tree.query(real_batch, k=nearest_k+1)

        for j in range(len(real_batch)):
            neighbors = indices[j, 1:]  # ignorar o próprio ponto
            fake_neighbors = np.sum(neighbors >= n_real)
            precision_scores.append(fake_neighbors / nearest_k)

    precision = np.mean(precision_scores) if precision_scores else 0.0

    # Calcular recall
    recall_scores = []
    batch_size = min(100, n_fake)

    for i in range(0, n_fake, batch_size):
        end_idx = min(i + batch_size, n_fake)
        fake_batch = all_features[n_real + i: n_real + end_idx]

        distances, indices = tree.query(fake_batch, k=nearest_k+1)

        for j in range(len(fake_batch)):
            neighbors = indices[j, 1:]  # ignorar o próprio ponto
            real_neighbors = np.sum(neighbors < n_real)
            recall_scores.append(real_neighbors / nearest_k)

    recall = np.mean(recall_scores) if recall_scores else 0.0

    return precision, recall

def compute_prdc_by_class(real_features, real_labels, fake_features, fake_labels, class_names, nearest_k=5):
    """
    Calcula PRDC para cada classe
    """
    results = {}

    for class_idx, class_name in enumerate(class_names):
        print(f"\n📊 Processando classe: {class_name}")

        # Filtrar features por classe
        real_mask = (real_labels == class_idx)
        fake_mask = (fake_labels == class_idx)

        real_features_class = real_features[real_mask]
        fake_features_class = fake_features[fake_mask]

        n_real = len(real_features_class)
        n_fake = len(fake_features_class)

        print(f"  Amostras: {n_real} real, {n_fake} fake")

        if n_real < nearest_k or n_fake < nearest_k:
            print(f"  ⚠️  Amostras insuficientes para cálculo (mínimo {nearest_k})")
            results[class_name] = {
                'precision': 0.0,
                'recall': 0.0,
                'n_real_samples': n_real,
                'n_fake_samples': n_fake,
                'status': 'insufficient_samples'
            }
            continue

        # Calcular PRDC para esta classe
        precision, recall = compute_prdc_for_class(real_features_class, fake_features_class, nearest_k)

        results[class_name] = {
            'precision': float(precision),
            'recall': float(recall),
            'n_real_samples': n_real,
            'n_fake_samples': n_fake,
            'status': 'success'
        }

        print(f"  ✅ Precision: {precision:.3f}, Recall: {recall:.3f}")

        # Limpeza de memória após cada classe
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return results

# ================= EXECUÇÃO PRINCIPAL =================
def main():
    print("🚀 Iniciando cálculo de P&R por classe")
    print("📁 Estrutura: Subpastas por classe")
    print(f"📍 Pasta real: {REAL_PATH}")
    print(f"📍 Pasta fake: {FAKE_PATH}")

    # Verificar pastas
    if not os.path.exists(REAL_PATH):
        raise ValueError(f"❌ Pasta real não encontrada: {REAL_PATH}")
    if not os.path.exists(FAKE_PATH):
        raise ValueError(f"❌ Pasta fake não encontrada: {FAKE_PATH}")

    # Analisar estrutura das pastas
    print("\n🔍 Analisando estrutura das pastas...")
    print("📂 Pasta REAL:")
    real_classes, real_class_images = get_subfolder_classes(REAL_PATH)

    print("\n📂 Pasta FAKE:")
    fake_classes, fake_class_images = get_subfolder_classes(FAKE_PATH)

    # Usar apenas classes comuns
    common_classes = list(set(real_classes) & set(fake_classes))
    print(f"\n✅ Classes comuns encontradas: {common_classes}")

    if not common_classes:
        raise ValueError("❌ Nenhuma classe comum encontrada entre real e fake!")

    # Verificar se as classes têm amostras suficientes
    for cls in common_classes:
        real_count = len(real_class_images[cls])
        fake_count = len(fake_class_images[cls])
        print(f"  {cls}: {real_count} real, {fake_count} fake")

    # Transformações
    transform = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    # Carregar datasets
    print("\n📦 Carregando datasets...")
    real_dataset = ImageFolderWithClasses(REAL_PATH, transform, MAX_SAMPLES_PER_CLASS)
    fake_dataset = ImageFolderWithClasses(FAKE_PATH, transform, MAX_SAMPLES_PER_CLASS)

    # Verificar se temos dados
    if len(real_dataset) == 0 or len(fake_dataset) == 0:
        raise ValueError("❌ Nenhuma imagem encontrada nos datasets!")

    # Criar dataloaders
    real_loader = torch.utils.data.DataLoader(real_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    fake_loader = torch.utils.data.DataLoader(fake_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    # Carregar modelo
    print("🧠 Carregando modelo ResNet18...")
    weights = ResNet18_Weights.IMAGENET1K_V1
    model = resnet18(weights=weights)
    model = torch.nn.Sequential(*list(model.children())[:-1])
    model = model.to(device)
    model.eval()

    # Extrair features
    print("🔧 Extraindo features reais...")
    real_features, real_labels = extract_features_with_labels(real_loader, model, device)

    print("🔧 Extraindo features fake...")
    fake_features, fake_labels = extract_features_with_labels(fake_loader, model, device)

    print(f"📊 Features reais: {real_features.shape}")
    print(f"📊 Features fake: {fake_features.shape}")

    # Calcular métricas por classe
    print("\n📈 Calculando métricas PRDC por classe...")
    results = compute_prdc_by_class(
        real_features, real_labels,
        fake_features, fake_labels,
        common_classes,
        nearest_k=5
    )

    # Salvar resultados
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    with open(os.path.join(OUTPUT_DIR, 'pr_by_class.json'), 'w') as f:
        json.dump(results, f, indent=4)

    # Mostrar resultados
    print("\n" + "="*80)
    print("🎯 RESULTADOS POR CLASSE")
    print("="*80)
    print(f"{'Classe':20s} | {'Precisão':8s} | {'Revocação':8s} | {'Real':6s} | {'Fake':6s} | Status")
    print("-" * 80)

    for class_name in common_classes:
        metrics = results[class_name]
        if metrics['status'] == 'success':
            print(f"{class_name:20s} | {metrics['precision']:8.3f} | {metrics['recall']:8.3f} | "
                  f"{metrics['n_real_samples']:6d} | {metrics['n_fake_samples']:6d} | ✅")
        else:
            print(f"{class_name:20s} | {'-':8s} | {'-':8s} | "
                  f"{metrics['n_real_samples']:6d} | {metrics['n_fake_samples']:6d} | ⚠️")

    print("="*80)
    print(f"💾 Resultados salvos em: {OUTPUT_DIR}")
    print("✅ Análise completa!")

if __name__ == "__main__":
    main()

Dispositivo: cuda
🚀 Iniciando cálculo de P&R por classe
📁 Estrutura: Subpastas por classe
📍 Pasta real: DATASET54000
📍 Pasta fake: DDPM-F64x4

🔍 Analisando estrutura das pastas...
📂 Pasta REAL:
Subpastas encontradas em DATASET54000: ['BULKCARRIER', 'CONTAINERSHIP', 'GENERALCARGO', 'OILPRODUCTSTANKER', 'PASSENGERSSHIP', 'TANKER', 'TRAWLER', 'TUG', 'VEHICLESCARRIER', 'YACHT']
  Classe 'BULKCARRIER': 5400 imagens
  Classe 'CONTAINERSHIP': 5400 imagens
  Classe 'GENERALCARGO': 5400 imagens
  Classe 'OILPRODUCTSTANKER': 5400 imagens
  Classe 'PASSENGERSSHIP': 5400 imagens
  Classe 'TANKER': 5400 imagens
  Classe 'TRAWLER': 5400 imagens
  Classe 'TUG': 5400 imagens
  Classe 'VEHICLESCARRIER': 5400 imagens
  Classe 'YACHT': 5400 imagens

📂 Pasta FAKE:
Subpastas encontradas em DDPM-F64x4: ['BULKCARRIER', 'CONTAINERSHIP', 'GENERALCARGO', 'OILPRODUCTSTANKER', 'PASSENGERSSHIP', 'TANKER', 'TRAWLER', 'TUG', 'VEHICLESCARRIER', 'YACHT']
  Classe 'BULKCARRIER': 5400 imagens
  Classe 'CONTAINERSHIP': 5